In [5]:
import os, json, math
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

In [10]:
load_dotenv(find_dotenv())

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.chat.completions.create(
    model=os.getenv("MODEL", "gpt-4o-mini"),
    messages=[
        {
            "role": "user",
            "content": "What are the Olympics?"
        }
    ],
    max_tokens=100,
)
print(response.choices[0].message.content)

The Olympics, formally known as the Olympic Games, is a major international multi-sport event that takes place every four years, featuring summer and winter sports. The Games are held under the auspices of the International Olympic Committee (IOC) and bring together athletes from around the world to compete in various disciplines.

### Key Points:

1. **History**: The Olympic Games have their origins in ancient Greece around 776 BC, where they were held in Olympia as part of a religious festival honoring Zeus.


# RTCF system prompt

In [11]:
system_prompt = """
Role:
You are an AI agent tutor and autonomous task assistant.

Task:
Use reasoning and tools to complete the user's task accurately.

Context:
You can use tools for calculation and AI-agent concept lookup.
Follow a plan → act → observe → refine process internally.
Do not reveal private chain-of-thought. Summarize your approach briefly.

Format:
Return:
1. Brief answer
2. Tool use summary
3. Reflection question for the learner
""".strip()

# Local tools

In [12]:
def calculator(expression: str) -> str:
    allowed_names = {
        "sqrt": math.sqrt,
        "pow": pow,
        "abs": abs,
        "round": round,
        "min": min,
        "max": max,
    }

    try:
        result = eval(expression, {"__builtins__": {}}, allowed_names)
        return str(result)
    except Exception as e:
        return f"Calculator error: {e}"


KNOWLEDGE_BASE = {
    "agent": "An AI agent is a system that uses a model to decide actions toward a goal, often using tools.",
    "tool calling": "Tool calling lets a model request that the application execute a function and return the result.",
    "memory": "Agent memory stores useful prior context, task state, or observations.",
    "planning": "Planning is when an agent breaks a goal into smaller steps before acting.",
    "rtcf": "RTCF stands for Role, Task, Context, and Format. It helps structure prompts clearly.",
}


def kb_lookup(query: str) -> str:
    query_lower = query.lower()

    matches = {
        key: value
        for key, value in KNOWLEDGE_BASE.items()
        if key in query_lower
    }

    if not matches:
        return json.dumps(
            {
                "not_found": "No exact matching entry found.",
                "available_topics": list(KNOWLEDGE_BASE.keys()),
            },
            indent=2,
        )

    return json.dumps(matches, indent=2)


TOOLS = {
    "calculator": calculator,
    "kb_lookup": kb_lookup,
}

# Tool schemas

In [13]:
tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a safe math expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Math expression, such as sqrt(144).",
                    }
                },
                "required": ["expression"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "kb_lookup",
            "description": "Look up short definitions about AI agents.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The AI-agent concept or topic to look up.",
                    }
                },
                "required": ["query"],
                "additionalProperties": False,
            },
        },
    },
]

# Agent loop

In [14]:
def run_tool(tool_name: str, tool_args: dict) -> str:
    if tool_name not in TOOLS:
        return f"Unknown tool: {tool_name}"

    try:
        return TOOLS[tool_name](**tool_args)
    except Exception as e:
        return f"Tool execution error: {e}"


def run_agent(user_task: str) -> str:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_task},
    ]

    first_response = client.chat.completions.create(
        model=os.getenv("MODEL", "gpt-4o-mini"),
        messages=messages,
        tools=tool_schema,
        tool_choice="auto",
        max_tokens=800,
    )

    assistant_message = first_response.choices[0].message

    if not assistant_message.tool_calls:
        return assistant_message.content

    messages.append(assistant_message)

    for tool_call in assistant_message.tool_calls:
        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        tool_result = run_tool(tool_name, tool_args)

        messages.append(
            {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result,
            }
        )

    final_response = client.chat.completions.create(
        model=os.getenv("MODEL", "gpt-4o-mini"),
        messages=messages,
        max_tokens=800,
    )

    return final_response.choices[0].message.content

# Agent Tasks

In [15]:
test_tasks = [
    "Explain what an AI agent is.",
    "What is RTCF prompt design?",
    "Explain tool calling and calculate sqrt(625).",
    "Explain how memory helps an agent complete multi-step tasks.",
]

for task in test_tasks:
    print("=" * 80)
    print("TASK:")
    print(task)
    print("\nAGENT RESPONSE:")
    print(run_agent(task))
    print()

TASK:
Explain what an AI agent is.

AGENT RESPONSE:
1. An AI agent is a system that leverages models to make decisions and take actions toward achieving specific goals, often utilizing various tools in the process.

2. I used an AI concept lookup tool to gather a concise definition of an AI agent.

3. How do you think AI agents can impact everyday tasks or decision-making?

TASK:
What is RTCF prompt design?

AGENT RESPONSE:
1. RTCF prompt design stands for Role, Task, Context, and Format. It provides a structured way to formulate prompts clearly for effective communication with AI.

2. I used a concept lookup tool to retrieve information about RTCF prompt design.

3. How do you see the RTCF structure impacting the clarity of your prompts when working with AI?

TASK:
Explain tool calling and calculate sqrt(625).

AGENT RESPONSE:
1. The square root of 625 is 25. 
2. I called a calculator tool to compute the square root and accessed a knowledge base tool for a definition of tool calling.
